# 19 — Active-site fingerprint

Residue × complex contact-persistence heatmap. Bright horizontal stripes are the canonical pharmacophore residues that every complex touches — the active-site fingerprint. Feeds the IFP (interaction-fingerprint) columns that appear in NB 20's correlation block and NB 21's active-vs-decoy separation.

_(Notebook auto-generated by `reproduce/split_monolith.py`. Self-contained: loads its data via `discovery9.io`, exports figures to `figures/19_active_site_fingerprint_figK.png`.)_


> **Reader guide.** *Experiment A3:* residue × complex contact-persistence heatmap.
>
> **Method:** per-residue contact fraction over the trajectory, arranged into a residue × complex
> matrix.
>
> **Reproducibility contract:** reads per-complex `hbond_timeseries.parquet` +
> `ifp_timeseries.parquet`; matrix persisted to `data/derived/35_ifp_matrix.csv`.

In [ ]:
# --- notebook preamble ---
NB_STEM = "35_active_site_fingerprint"

import sys, os, json, glob
from pathlib import Path

# Make the in-repo src package importable without an install
# find repo root robustly (walks up until pyproject.toml)
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent and not (_repo_root / 'pyproject.toml').is_file():
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from discovery9.style import apply_style, NAVY, GOLD, GREY, GREY_DASH as GREYD, CREAM, WHITE, ACTIVE, DECOY, WARN
from discovery9.paths import ROOT, RAW, DERIVED, EXTERNAL, FIGURES, TABLES, GBSA_STUDY
from discovery9.io    import load_features, load_gbsa, load_gbsa_all, load_metadata, load_bedroc_matrix, load_bedroc_all_combos, load_per_complex_analysis
from discovery9.metrics import bedroc, bedroc_per_target, rank_fuse
apply_style()

# --- fig-capture hook (iter-3 fix) ---
_SAVED_FIGS = globals().setdefault('_SAVED_FIGS', [])
_orig_figure = plt.figure
_orig_subplots = plt.subplots
def _figure_capture(*a, **kw):
    fig = _orig_figure(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig
def _subplots_capture(*a, **kw):
    fig, ax = _orig_subplots(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig, ax
plt.figure = _figure_capture
plt.subplots = _subplots_capture

# Legacy monolith aliases:
ACTIVE_C, DECOY_C = ACTIVE, DECOY

# Default: load the master feature table (with ligand-chem descriptors when available)
df = load_features(with_ligand_chem=True)
print(f'features.parquet: {len(df)} complexes × {df.shape[1]} columns  ·  targets: {df.target.nunique()}')


## 7. Active-site fingerprint — residue × complex persistence matrix

One heatmap per target (edit `TARGET` in the code cell). Rows are active-site residues, columns are the 30 complexes. Actives are grouped left with a GOLD strip above their columns. Cell value = fraction of frames where that residue is in contact (< 4.5 Å) with any ligand heavy atom.

Read column-wise for how many contacts the ligand makes. Read row-wise for which residues are canonical for the pocket.


In [ ]:

def load_contact_matrix(target):
    frames = []
    for _, row in df[df.target == target].iterrows():
        cp_p = os.path.join(str(RAW / 'complex_analyses'), target, row.complex_id, 'contacts_persistence.tsv')
        if not os.path.exists(cp_p): continue
        cp = pd.read_csv(cp_p, sep='\t')
        cp['complex_id'] = row.complex_id
        frames.append(cp)
    if not frames: return None, None
    all_cp = pd.concat(frames, ignore_index=True)
    pivot = all_cp.pivot_table(index=['resid','resname'], columns='complex_id', values='persistence', fill_value=0)
    if 'is_active' in df:
        cid_active = df.set_index('complex_id')['is_active'].to_dict()
        cols = sorted(pivot.columns, key=lambda c: (not bool(cid_active.get(c, False)), c))
        pivot = pivot[cols]
        active_mask = [bool(cid_active.get(c, False)) for c in pivot.columns]
    else:
        active_mask = None
    return pivot, active_mask

TARGET = '4QB3'  # ← edit
pivot, active_mask = load_contact_matrix(TARGET)
if pivot is None:
    print(f'no contact data for {TARGET}')
else:
    fig, ax = plt.subplots(figsize=(14, max(4, 0.28*len(pivot))))
    im = ax.imshow(pivot.values, cmap='YlOrBr', vmin=0, vmax=1, aspect='auto')
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f'{int(r)} {n}' for r,n in pivot.index], fontsize=8)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([c[:8] for c in pivot.columns], rotation=90, fontsize=6)
    if active_mask is not None:
        for i, act in enumerate(active_mask):
            if act:
                ax.add_patch(plt.Rectangle((i-0.4, -1.2), 0.8, 0.6, color=GOLD, clip_on=False))
    cbar = plt.colorbar(im, ax=ax, label='contact persistence', shrink=0.7)
    cbar.ax.yaxis.label.set_color(NAVY)
    ax.set_title(f'{TARGET} · residue × complex contact-persistence  (GOLD strip = active)')

**What the patterns mean.** Bright **horizontal stripes** are the canonical binding-site residues — the pharmacophore MD picks up that static docking doesn't. Bright **vertical stripes** over a GOLD-marked column mean an active engages the full pocket. Dark GOLD-marked columns are actives that either escaped or hit only a sub-pocket. Non-active columns that light up on canonical residues are the hard decoys — they mimic the interaction pattern and would rank high on any contact-only score.


In [ ]:
# --- export every figure produced in this notebook (iter-3 fix) ---
try:
    FIGURES.mkdir(parents=True, exist_ok=True)
except NameError:
    from discovery9.paths import FIGURES
    FIGURES.mkdir(parents=True, exist_ok=True)
try:
    _cream = CREAM
except NameError:
    from discovery9.style import CREAM as _cream
figs = list(globals().get('_SAVED_FIGS', []))
# fallback: any figures still open in the backend
for num in plt.get_fignums():
    f = plt.figure(num)
    if f not in figs:
        figs.append(f)
saved = []
for i, fig in enumerate(figs, start=1):
    out = FIGURES / f"{NB_STEM}_fig{i}.png"
    try:
        fig.savefig(out, bbox_inches='tight', dpi=300, facecolor=_cream)
    except Exception as e:
        print(f'  WARN: failed to save fig{i}: {e}')
        continue
    saved.append(str(out.name))
print(f'saved {len(saved)} figures:')
for s in saved:
    print(' ', s)
